In [0]:
# INGESTION — physical_itens_venda_caixa
# Squad 3 — Batch Ecommerce

# Acesso ao config e utils

%run "/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/Squad3/luiz-portacio/config/00_config.ipynb"
%run "/Workspace/Repos/luizhpdatasci@gmail.com/merca-data-platform/Squad3/luiz-portacio/utils/00_utils.ipynb"


In [0]:
# Listar arquivos

arquivos = listar_arquivos(adls_client, container)

In [0]:
# Ler arquivo grande

# Célula 3 — Ler arquivo em chunks (277MB)
df_itens = ler_csv_chunks(
    adls_client,
    container,
    "physical_itens_venda_caixa.csv",
    chunk_size=50000
)

In [0]:
# Análise Exploratória

# Célula 4 — Análise Exploratória
print("=" * 50)
print("📊 ANÁLISE EXPLORATÓRIA — physical_itens_venda_caixa")
print("=" * 50)

print(f"\n📐 Shape: {df_itens.shape}")
print(f"   {df_itens.shape[0]} linhas | {df_itens.shape[1]} colunas")

print("\n📋 Colunas e tipos:")
print(df_itens.dtypes)

print("\n❓ Nulos por coluna:")
print(df_itens.isnull().sum())

print(f"\n🔁 Duplicatas: {df_itens.duplicated().sum()}")

print("\n📈 Estatísticas:")
df_itens.describe()

In [0]:
# Primeiras Linhas com spark

print("👀 Primeiras 10 linhas:")
df_itens.head(10)


In [0]:
# Tratamentos

print("=" * 50)
print("🔧 TRATAMENTOS — physical_itens_venda_caixa")
print("=" * 50)

# ══════════════════════════════════════
# 1. DUPLICATAS — já confirmado que não há
# ══════════════════════════════════════
total_antes = len(df_itens)
df_itens    = df_itens.drop_duplicates()
print(f"✅ Duplicatas removidas: {total_antes - len(df_itens)} linhas")

# ══════════════════════════════════════
# 2. NULOS — já confirmado que não há
# mas mantemos por segurança
# ══════════════════════════════════════
colunas_texto = df_itens.select_dtypes(include='object').columns
for col in colunas_texto:
    df_itens[col] = df_itens[col].fillna('desconhecido').str.strip()
print("✅ Nulos em texto tratados")

colunas_num = df_itens.select_dtypes(include='number').columns
for col in colunas_num:
    df_itens[col] = df_itens[col].fillna(0)
print("✅ Nulos em números tratados")

# ══════════════════════════════════════
# 3. IDs — converter para string
# ══════════════════════════════════════
df_itens['id_item_venda'] = df_itens['id_item_venda'].astype(str)
df_itens['id_transacao']  = df_itens['id_transacao'].astype(str).str.strip()
print("✅ id_item_venda e id_transacao convertidos para string")

# ══════════════════════════════════════
# 4. CODIGO_BARRAS — limpar e padronizar
# ══════════════════════════════════════
df_itens['codigo_barras_produto'] = (
    df_itens['codigo_barras_produto']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', '', regex=True)  # remove espaços internos
)
print("✅ codigo_barras_produto padronizado")

# ══════════════════════════════════════
# 5. QUANTIDADE — verificar decimais e negativos
# ══════════════════════════════════════
# Verifica se há quantidades com decimais
decimais = (df_itens['quantidade'] % 1 != 0).sum()
print(f"\n⚠️  Quantidades com decimais: {decimais}")

# Corrige negativos
df_itens['quantidade'] = df_itens['quantidade'].abs()
print("✅ quantidade — negativos corrigidos")

# ══════════════════════════════════════
# 6. VALORES — verificar negativos
# ══════════════════════════════════════
for col in ['preco_unitario_registro', 'valor_total_item']:
    negativos = (df_itens[col] < 0).sum()
    if negativos > 0:
        df_itens[col] = df_itens[col].abs()
        print(f"✅ {col} — {negativos} negativos corrigidos")
    else:
        print(f"✅ {col} — sem valores negativos")

# ══════════════════════════════════════
# 7. VERIFICAR CONSISTÊNCIA
# valor_total_item deve ser próximo de
# quantidade * preco_unitario_registro
# ══════════════════════════════════════
df_itens['valor_calculado'] = (
    df_itens['quantidade'] * df_itens['preco_unitario_registro']
).round(2)

df_itens['valor_total_item'] = df_itens['valor_total_item'].round(2)

inconsistencias = (
    abs(df_itens['valor_calculado'] - df_itens['valor_total_item']) > 0.05
).sum()

print(f"\n⚠️  Inconsistências valor_total vs calculado: {inconsistencias}")

# Remove coluna auxiliar
df_itens = df_itens.drop(columns=['valor_calculado'])
print("✅ Coluna auxiliar removida")

# ══════════════════════════════════════
# RESULTADO FINAL
# ══════════════════════════════════════
print(f"\n📐 Shape final: {df_itens.shape}")
print("\n👀 Amostra final:")
df_itens.head()

In [0]:
# Salvar no SQL Server em lotes

# Célula 7 — Salvar no SQL Server em lotes
salvar_tabela_lotes(
    df_itens,
    nome_tabela  = "physical_itens_venda_caixa",
    lote_size    = 5000,       # ← reduzido de 10k para 5k
    max_tentativas = 3         # ← retry automático
)

In [0]:
# Verificar Dados Salvos

df_verificacao = consultar_tabela("physical_itens_venda_caixa")
print(f"✅ Verificação concluída!")
print(f"   Linhas no banco: {len(df_verificacao)}")
df_verificacao.head()